# Category-filtered object RAG: design and plan

Question + category → mapped active documents → eligible table/outer-row vectors → cosine ranking → exact cited evidence → Ollama code answer.

Reuse the existing index without ingestion or checkpoint changes. Match the combined and V4.0/V5.0 workbook mappings through explicit corpus-root aliases, retain corpus and revision labels, and report missing coverage. Traverse row-group wrappers for outer rows; exclude cells and deeper rows from vector candidates. Validate identity, embedding configuration and dimensions; query eligible IDs before ranking.

Implementation: configure paths and services; load categories/mappings; open the catalog read-only and existing collection; inspect coverage; retrieve and inspect exact evidence; generate version-specific fenced snippets with citations; close resources. Defaults: supplied index, top-k 8, 24,000 context characters, Ollama RAG model falling back to the summary model. Complete blocks exceeding the budget are reported as omitted. Unsupported coding requests must receive an insufficient-evidence answer.

Validation: offline fake-provider and ephemeral-index tests, notebook validation and cell compilation, and read-only real catalog coverage inspection. No corpus rebuilds, summary requests or checkpoint resets.


Every answer includes a separate source-attributed summary of the retrieved objects, regardless of whether the coding request is supported.

## Application configuration

Reusable retrieval logic lives in [01.rag.object.ipynb](01.rag.object.ipynb), exported as `ribosome.rag.object`. Configuration and client lifecycle stay here.

In [ ]:
# | notest
import os
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv
from IPython.display import Markdown, display
from ribosome.rag.object import ObjectRAG

PROJ_ROOT = Path(os.getenv("PROJ_ROOT", Path.cwd())).resolve()
while not (PROJ_ROOT / "pyproject.toml").exists() and PROJ_ROOT != PROJ_ROOT.parent:
    PROJ_ROOT = PROJ_ROOT.parent
load_dotenv(PROJ_ROOT / ".env", override=False)
# INDEX_DIR = Path(os.getenv("OBJECT_RAG_INDEX", "db/object_embeddings/qwen3-embedding-8b-20261022-152558-151254"))
INDEX_DIR = Path(os.getenv("OBJECT_RAG_INDEX", "db/object_embeddings/qwen3-embedding-8b"))
if not INDEX_DIR.is_absolute(): INDEX_DIR = PROJ_ROOT / INDEX_DIR
EMBED_MODEL = os.getenv("OLLAMA_EMBED_MODEL", "qwen3-embedding:8b")
HOSTS = sorted(set(x.strip().rstrip("/") for x in os.getenv("OLLAMA_EMBED_HOSTS", "http://localhost:11434,http://172.27.74.16:11434").split(",") if x.strip()))
RAG_MODEL = os.getenv("OLLAMA_RAG_MODEL") or os.getenv("OLLAMA_SUMMARY_MODEL", "")
RAG_HOST = os.getenv("OLLAMA_RAG_HOST") or os.getenv("OLLAMA_HOST", "http://localhost:11434")
MAX_CONTEXT_CHARS = 24000
REQUEST_TIMEOUT = 300
MAX_OUTPUT_TOKENS = 2048



## Open existing index and services
Run explicitly; no embedding or generation occurs in this cell.

In [ ]:
# | notest
import chromadb
from ollama import AsyncClient
if not (INDEX_DIR / 'catalog.sqlite3').is_file() or not (INDEX_DIR / 'chroma').is_dir():
    raise FileNotFoundError(INDEX_DIR)
headers = {'Authorization': 'Bearer '+os.environ['OLLAMA_API_KEY']} if os.getenv('OLLAMA_API_KEY') else {}
embedding_client = AsyncClient(host=HOSTS[0], headers=headers, timeout=REQUEST_TIMEOUT)
generation_client = AsyncClient(host=RAG_HOST, headers=headers, timeout=REQUEST_TIMEOUT)
collection = chromadb.PersistentClient(path=str(INDEX_DIR/'chroma')).get_collection('object_summaries', embedding_function=None)
app = ObjectRAG(
    INDEX_DIR, collection, embedding_client, generation_client,
    mapping_dir=PROJ_ROOT / 'assets/coding_categories',
    embed_model=EMBED_MODEL,
    embedding_config=dict(provider='ollama-pool', hosts=HOSTS),
    rag_model=RAG_MODEL, max_context_chars=MAX_CONTEXT_CHARS,
    max_output_tokens=MAX_OUTPUT_TOKENS,
)
async def retrieve(question, category, *, top_k=8):
    return await app.retrieve(question, category, top_k=top_k)
async def answer(question, category, *, top_k=8):
    return await app.answer(question, category, top_k=top_k)


## Category coverage

In [ ]:
# | notest
display(pd.DataFrame(app.categories.items(), columns=['category','chinese_name']))
category = 'general_operation'  # 'teaching_free_welding'
question = '如何设置用户坐标号?'  # '如何编写免示教焊接程序？'
display(app.snapshot(category)[2])

In [ ]:
category

## Retrieve and inspect exact evidence

In [ ]:
# | notest
result = await retrieve(question, category)
display(pd.DataFrame(result['hits']))
print(result['context'])
print('Omitted:', result['omissions'])

## Retrieved content summary and coding answer

Always display a source-attributed summary of the found objects, even when their content cannot support the requested code. The summary uses stored object summaries and remains available if generation fails. With no hits, explicitly report that no relevant indexed content was found.

In [ ]:
# | notest
try:
    result = await answer(question, category)
finally:
    # Retain the retrieved-content summary even if the answer model fails.
    if app.last_result is not None and 'retrieved_summary' in app.last_result:
        display(Markdown('## Retrieved content summary\n\n' + app.last_result['retrieved_summary']))
display(Markdown('## Coding answer\n\n' + result['answer']))
display(result['coverage'])

## Cleanup
Run after completion or interruption.

In [ ]:
# | notest
try:
    if 'app' in globals(): app.close()
finally:
    for client_name in ('embedding_client', 'generation_client'):
        if client_name in globals(): await globals()[client_name]._client.aclose()